In [3]:
## Masked prediction shift
'''
For each **candidate** in a **time period**, we mask the candidate span in its **corpus sentence** and read off a masked LM’s **top‑k** predictions. If **taboo‑related** tokens appear more often in those predictions in **later** periods, that supports a shift of contextual usage toward the taboo domain.

### Upstream output types (this repo)

**`first_pass.py` → JSONL (`PipelineConfig.output_path`, default `first_pass_results.jsonl`)**  
Each line is `EuphemismCandidate` as a dict: `text` (extracted phrase), `context` (full sentence), `taboo_anchor`, `taboo_category`, `similarity_score`, `phrase_extraction_method`, `phrase_drop_score`, `timestamp`, `source_url`, `source`, optional `context_window_size`, `all_anchor_matches`.

**`second_pass.py` → JSONL (`SecondPassConfig.output_path`, default `second_pass_results.jsonl`)**  
Each line is `EuphemismInstance`: `phrase`, `canonical_phrase`, `sentence`, `before_context`, `after_context`, `char_offset_start`, `char_offset_end`, `taboo_anchors`, `primary_category`, `first_pass_similarity`, `first_pass_phrase_drop`, `timestamp`, `source_url`, `source`, `match_mode`, `is_variant`.

This notebook uses **second‑pass** rows so we mask the **exact matched span** via character offsets. You can adapt the same masking code to **first‑pass** rows using `text` + `context` (find span of `text` in `context`).
'''

'\nFor each **candidate** in a **time period**, we mask the candidate span in its **corpus sentence** and read off a masked LM’s **top‑k** predictions. If **taboo‑related** tokens appear more often in those predictions in **later** periods, that supports a shift of contextual usage toward the taboo domain.\n\n### Upstream output types (this repo)\n\n**`first_pass.py` → JSONL (`PipelineConfig.output_path`, default `first_pass_results.jsonl`)**  \nEach line is `EuphemismCandidate` as a dict: `text` (extracted phrase), `context` (full sentence), `taboo_anchor`, `taboo_category`, `similarity_score`, `phrase_extraction_method`, `phrase_drop_score`, `timestamp`, `source_url`, `source`, optional `context_window_size`, `all_anchor_matches`.\n\n**`second_pass.py` → JSONL (`SecondPassConfig.output_path`, default `second_pass_results.jsonl`)**  \nEach line is `EuphemismInstance`: `phrase`, `canonical_phrase`, `sentence`, `before_context`, `after_context`, `char_offset_start`, `char_offset_end`, `

In [2]:
# pip install torch transformers pandas matplotlib
# (Use a GPU runtime if available for speed.)

from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

# --- Paths & model ---
REPO_ROOT = Path(".").resolve()
SECOND_PASS_JSONL = REPO_ROOT / "second_pass_results.jsonl"
TABOO_WORDS_PY = REPO_ROOT / "taboo_words_refined.py"

MODEL_NAME = "bert-base-uncased"
TOP_K = 15
# Cap rows for development; set None to use all loaded rows (can be slow).
MAX_INSTANCES: int | None = 2000
# Optional: max sentences per (canonical_phrase, period) to balance periods.
MAX_PER_GROUP: int | None = 50

# Time bucketing: "year" | "decade"
TIME_BUCKET = "year"

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def load_taboo_vocab(path: Path) -> set[str]:
    """Flatten TABOO_ANCHORS from taboo_words_refined.py into normalized strings."""
    ns: dict = {}
    with open(path, "r") as f:
        exec(f.read(), ns)
    anchors = ns["TABOO_ANCHORS"]
    out: set[str] = set()
    for _cat, words in anchors.items():
        for w in words:
            w = str(w).strip().lower()
            if w:
                out.add(w)
                for part in re.split(r"[\s\-]+", w):
                    if len(part) > 2:
                        out.add(part)
    return out


def parse_timestamp(raw: str) -> datetime | None:
    if not raw or not str(raw).strip():
        return None
    s = str(raw).strip()
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except ValueError:
        pass
    for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%Y/%m/%d"):
        try:
            return datetime.strptime(s[:19], fmt) if len(s) >= 10 else datetime.strptime(s, fmt)
        except ValueError:
            continue
    m = re.search(r"(20\d{2}|19\d{2})", s)
    if m:
        return datetime(int(m.group(1)), 1, 1)
    return None


def bucket_label(dt: datetime, mode: str) -> str:
    if mode == "decade":
        d = dt.year - (dt.year % 10)
        return f"{d}s"
    return str(dt.year)


def prediction_hits_taboo(decoded_topk_strings: list[str], taboo_vocab: set[str]) -> bool:
    """True if any decoded prediction token matches the taboo lexicon."""
    for s in decoded_topk_strings:
        for word in re.findall(r"[a-zA-Z]+", s.lower()):
            if word in taboo_vocab:
                return True
    return False

In [6]:
def mask_phrase_spans(
    tokenizer,
    sentence: str,
    char_start: int,
    char_end: int,
    device: torch.device,
) -> tuple[torch.Tensor, list[int]]:
    """
    Replace all tokenizer pieces overlapping [char_start, char_end) with [MASK].
    Returns input_ids (1, seq) and indices of masked positions.
    """
    enc = tokenizer(
        sentence,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=512,
    )
    input_ids = enc["input_ids"].to(device).clone()
    off = enc["offset_mapping"][0]
    off_list = off.tolist() if hasattr(off, "tolist") else list(off)
    mask_id = tokenizer.mask_token_id
    masked_positions: list[int] = []
    for i, (s, e) in enumerate(off_list):
        if s == 0 and e == 0:
            continue
        if s >= char_end:
            break
        if e <= char_start:
            continue
        input_ids[0, i] = mask_id
        masked_positions.append(i)
    if not masked_positions:
        return input_ids, []
    return input_ids, masked_positions


@torch.inference_mode()
def topk_at_masked_positions(
    model,
    tokenizer,
    input_ids: torch.Tensor,
    masked_positions: list[int],
    k: int,
) -> list[str]:
    """Union of top-k predicted tokens across all [MASK] positions (single-string labels)."""
    out: list[str] = []
    if not masked_positions:
        return out
    logits = model(input_ids).logits[0]
    for pos in masked_positions:
        topv, topi = torch.topk(logits[pos], k=min(k, logits.shape[-1]))
        for tid in topi.tolist():
            tok = tokenizer.decode([tid]).strip()
            if tok:
                out.append(tok)
    return out


def row_taboo_hit(
    model,
    tokenizer,
    taboo_vocab: set[str],
    sentence: str,
    c0: int,
    c1: int,
    k: int,
    device: torch.device,
) -> bool:
    input_ids, positions = mask_phrase_spans(tokenizer, sentence, c0, c1, device)
    if not positions:
        return False
    toks = topk_at_masked_positions(model, tokenizer, input_ids, positions, k)
    return prediction_hits_taboo(toks, taboo_vocab)

In [7]:
def load_second_pass(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)
            except json.JSONDecodeError:
                continue
            rows.append(o)
    return pd.DataFrame(rows)


taboo_vocab = load_taboo_vocab(TABOO_WORDS_PY)
print(f"Loaded {len(taboo_vocab)} taboo-related vocabulary items (with word parts).")

if not SECOND_PASS_JSONL.is_file():
    raise FileNotFoundError(
        f"Missing {SECOND_PASS_JSONL}. Run second_pass.py or set SECOND_PASS_JSONL to your JSONL path."
    )

df_raw = load_second_pass(SECOND_PASS_JSONL)
print(f"Second-pass rows: {len(df_raw)}")
print("Columns:", list(df_raw.columns))

# Parse time and bucket
df_raw["_dt"] = df_raw["timestamp"].map(parse_timestamp)
df = df_raw[df_raw["_dt"].notna()].copy()
df["period"] = df["_dt"].map(lambda d: bucket_label(d, TIME_BUCKET))
if MAX_INSTANCES is not None and len(df) > MAX_INSTANCES:
    df = df.sample(n=MAX_INSTANCES, random_state=42)
if MAX_PER_GROUP is not None:
    df = df.groupby(["canonical_phrase", "period"], group_keys=False).head(
        MAX_PER_GROUP
    )
print(f"Rows after time filter + caps: {len(df)}")
df[["canonical_phrase", "period", "sentence"]].head(3)

Loaded 125 taboo-related vocabulary items (with word parts).


FileNotFoundError: Missing /Users/kareenamehta/Documents/vscode/ml_proj/Emerging-and-New-Euphemism-Detection/second_pass_results.jsonl. Run second_pass.py or set SECOND_PASS_JSONL to your JSONL path.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

records: list[dict] = []
for _, row in df.iterrows():
    sent = str(row["sentence"])
    c0, c1 = int(row["char_offset_start"]), int(row["char_offset_end"])
    hit = row_taboo_hit(
        model,
        tokenizer,
        taboo_vocab,
        sent,
        c0,
        c1,
        TOP_K,
        device,
    )
    records.append(
        {
            "canonical_phrase": row["canonical_phrase"],
            "period": row["period"],
            "taboo_in_topk": hit,
            "primary_category": row.get("primary_category", ""),
        }
    )

res = pd.DataFrame(records)
agg = (
    res.groupby(["canonical_phrase", "period"], as_index=False)
    .agg(taboo_rate=("taboo_in_topk", "mean"), n=("taboo_in_topk", "size"))
    .sort_values(["canonical_phrase", "period"])
)
agg.head(20)

In [ ]:
# Line plot: taboo hit rate over periods for each phrase (limit to phrases with enough n)
MIN_N_PER_PERIOD = 3
wide = agg.pivot(index="period", columns="canonical_phrase", values="taboo_rate")
counts = (
    res.groupby(["canonical_phrase", "period"])
    .size()
    .unstack(fill_value=0)
)
# keep columns where every nonzero period has at least MIN_N_PER_PERIOD
phrases_ok = [
    c
    for c in wide.columns
    if (counts[c] >= MIN_N_PER_PERIOD).sum() >= 2
]
if not phrases_ok:
    phrases_ok = list(wide.columns)[:8]

fig, ax = plt.subplots(figsize=(10, 5))
for phrase in phrases_ok[:12]:
    y = wide[phrase].dropna()
    if y.empty:
        continue
    ax.plot(y.index.astype(str), y.values, marker="o", label=phrase[:40])
ax.set_xlabel("Period")
ax.set_ylabel(f"Fraction of contexts with taboo token in top-{TOP_K}")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("Masked LM: taboo-related predictions over time (by candidate)")
plt.tight_layout()
plt.show()